# GLP-1 Start Guide: analysis notebook

This notebook re-does the app's week analysis in SQL (DuckDB), separately from the TypeScript engine, so the two can check each other.

**The question.** Does resting heart rate, HRV, recovery or sleep performance follow a repeatable shape across the 7 days of each weekly injection cycle?

**The method, in plain language.**

1. **Your usual week.** Most people's numbers already differ by weekday (late Friday nights, early Monday alarms). For each metric, take the average for each weekday over the 12 weeks before the first dose. Each weekday needs at least 6 values.
2. **Adjust for it.** Subtract that weekday's usual value from every day. What remains is "above or below your usual for that weekday".
3. **Cut into injection cycles.** Each cycle runs from one injection to the next. `d = 0` is the first wake-up after the injection, and `d = 6` the last morning before the next one. Cycles that aren't 7 days long, or that have fewer than 5 days with data, are left out.
4. **Center each cycle.** Subtract the cycle's own average. This removes slow change across weeks (weight loss, dose level, season), so what's left is the shape *within* the week.
5. **Average across cycles.** `P(d)` is the average of those centered values on each day `d`. It is a relative shape that averages to zero across the week, not a level.

The **raw view** skips step 2. Comparing raw with adjusted shows the *weekday trap*: if injections always fall on the same weekday, a weekday habit looks like a drug pattern in the raw view.

**What this notebook doesn't do.** Point estimates only. The app's intervals (bootstrap over cycles and pre-drug weeks, t intervals with min-df) and its "clear pattern" rule live in the TypeScript engine. Persona A's point estimates must match `engine_results.json` to rounding.

All values that change results are in the params cell, which must equal `src/config/analysis.ts` (the cross-check asserts it). The SQL lives in `sql/*.sql`.


In [ ]:
import json
import re
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Works whether the kernel starts in notebook/ (VS Code default) or the repo root.
NB = Path.cwd() if (Path.cwd() / "sql").is_dir() else Path.cwd() / "notebook"
SQL_DIR = NB / "sql"
SYNTHETIC = NB / "data" / "synthetic"
EXPORT = NB / "data" / "export"

WEEKDAYS = ["Sun", "Mon", "Tue", "Wed", "Thu", "Fri", "Sat"]
pd.set_option("display.precision", 3)

## Parameters

Mirrors `analysisConfig` in `src/config/analysis.ts` (set by Prakhar, 2026-10-12, interval 2026-10-13). Don't edit values here without changing the config; the persona A cross-check fails if they differ.

In [ ]:
PARAMS = {
    "preDrugWindow": {"kind": "lastWeeks", "weeks": 12},
    "minPreDrugWeeksPerWeekday": 6,
    "minCycles": 6,
    "excludeWeeksAfterStart": 0,
    "excludeWeeksAfterDoseStep": 0,
    "dayZeroRule": {"morning": 1, "evening": 1},
    "minDaysPerCycle": 5,
    "irregularCycleRule": "drop",
    "missedInjectionRule": "dropCycle",
    "calibratingRule": "exclude",
    # Not used here (point estimates only); kept so the config check covers everything.
    "uncertaintyMethod": {
        "kind": "bootstrapCycles",
        "resamples": 10000,
        "seed": 1,
        "level": 0.95,
        "interval": "tMinDf",
    },
    "clearPatternRule": {"minDaysIntervalExcludesZero": 1, "correction": "acrossDaysAndMetrics"},
    "metricsShown": ["restingHr", "hrvMs", "recovery", "sleepPerformance"],
    "rollingWindowDays": 14,
    "rollingMinDays": 10,
    "rollingAlignment": "trailing",
    "historyDaysToFetch": 365,
}

## Pipeline

`run_sql` runs one `.sql` file statement by statement, binding only the `$params` each statement uses. `require` plays the role of `requireConfig()`: a `None` value stops the run instead of falling back to a default.

In [ ]:
class ConfigNotSetError(Exception):
    pass


def require(cfg, key):
    if cfg.get(key) is None:
        raise ConfigNotSetError(f"{key} is not set")
    return cfg[key]


def run_sql(con, name, **params):
    text = (SQL_DIR / name).read_text()
    # Drop comment lines before splitting, so a ';' in a comment can't split a statement.
    text = "\n".join(line for line in text.splitlines() if not line.lstrip().startswith("--"))
    for stmt in (s.strip() for s in text.split(";")):
        if not stmt:
            continue
        names = sorted(set(re.findall(r"\$(\w+)", stmt)))
        con.execute(stmt, {n: params[n] for n in names})


def register_inputs(con, inputs):
    # Member inputs (as in <persona>_inputs.json) as small tables for 04_injection_cycles.sql.
    con.execute("CREATE OR REPLACE TABLE inj_segments (from_date DATE, weekday INTEGER)")
    con.execute("CREATE OR REPLACE TABLE inj_missed (date DATE)")
    con.execute("CREATE OR REPLACE TABLE inj_moved (scheduled DATE, actual DATE)")
    con.execute("CREATE OR REPLACE TABLE dose_steps (date DATE, mg DOUBLE)")
    for s in inputs["injectionWeekdays"]:
        con.execute("INSERT INTO inj_segments VALUES (?, ?)", [s["from"], s["weekday"]])
    for d in inputs["missedInjectionDates"]:
        con.execute("INSERT INTO inj_missed VALUES (?)", [d])
    for m in inputs["movedInjections"]:
        con.execute("INSERT INTO inj_moved VALUES (?, ?)", [m["scheduled"], m["actual"]])
    for s in inputs["doseSteps"]:
        con.execute("INSERT INTO dose_steps VALUES (?, ?)", [s["date"], s["mg"]])


def assert_one_row_per_date(con):
    # Two rows on one date are flagged, never merged (CLAUDE.md, "Daily row").
    dupes = con.execute(
        "SELECT date FROM daily GROUP BY date HAVING count(DISTINCT cycle_id) > 1 ORDER BY date"
    ).fetchall()
    assert not dupes, f"{len(dupes)} dates have more than one row: {[str(d[0]) for d in dupes]}"


def run_pipeline(con, inputs, view, cfg=PARAMS):
    # Steps 1-5 on the `daily` table already in `con`. view is "adjusted" or "raw".
    window = require(cfg, "preDrugWindow")
    irregular = require(cfg, "irregularCycleRule")
    missed = require(cfg, "missedInjectionRule")
    if irregular != "drop":
        raise NotImplementedError(f'irregularCycleRule "{irregular}"')
    if missed != "dropCycle":
        raise NotImplementedError(f'missedInjectionRule "{missed}"')

    register_inputs(con, inputs)
    first_dose = inputs["firstDoseDate"]
    run_sql(
        con,
        "02_predrug_weekday_means.sql",
        metrics=require(cfg, "metricsShown"),
        calibrating_rule=require(cfg, "calibratingRule"),
        first_dose=first_dose,
        window_kind=window["kind"],
        window_weeks=window.get("weeks"),
        min_per_weekday=require(cfg, "minPreDrugWeeksPerWeekday"),
    )
    run_sql(con, "03_adjusted.sql", view=view)
    run_sql(
        con,
        "04_injection_cycles.sql",
        first_dose=first_dose,
        day_zero_offset=require(cfg, "dayZeroRule")[inputs["injectionTime"]],
        exclude_after_start=require(cfg, "excludeWeeksAfterStart"),
        exclude_after_step=require(cfg, "excludeWeeksAfterDoseStep"),
        min_days=require(cfg, "minDaysPerCycle"),
    )
    run_sql(con, "05_centered.sql")
    run_sql(con, "06_profile.sql", view=view, min_cycles=require(cfg, "minCycles"))

    order = cfg["metricsShown"]
    means = con.sql("SELECT * FROM predrug_means").df()
    profile = con.sql("SELECT * FROM profile").df()
    return {
        "view": view,
        "weekday_means": means.pivot(index="metric", columns="weekday", values="mean").reindex(order),
        "weekday_counts": means.pivot(index="metric", columns="weekday", values="n").reindex(order),
        "status": con.sql("SELECT * FROM profile_status").df().set_index("metric").reindex(order),
        "dropped": con.sql(
            "SELECT coalesce(drop_reason, 'kept by schedule rules') AS reason, count(*) AS cycles FROM cycles GROUP BY 1 ORDER BY 1"
        ).df(),
        "profile": profile.pivot(index="metric", columns="d", values="estimate").reindex(order),
    }


def analyze(load_daily, inputs):
    # Runs both views on a fresh in-memory database. load_daily(con) builds `daily`.
    out = {}
    for view in ("adjusted", "raw"):
        con = duckdb.connect()
        load_daily(con)
        assert_one_row_per_date(con)
        out[view] = run_pipeline(con, inputs, view)
        con.close()
    return out


def show(result):
    adj, raw = result["adjusted"], result["raw"]
    display(adj["status"])
    print("Injection cycles by rule (minDaysPerCycle is applied per metric afterwards; see cycles above):")
    display(adj["dropped"])
    print("Pre-drug weekday means (0 = Sunday):")
    display(adj["weekday_means"].rename(columns=dict(enumerate(WEEKDAYS))))
    print("Pre-drug values per weekday:")
    display(adj["weekday_counts"].rename(columns=dict(enumerate(WEEKDAYS))))
    print("P(d), adjusted:")
    display(adj["profile"])
    print("P(d), raw:")
    display(raw["profile"])

## Charts

Small multiples, one panel per metric, each on its own scale. Adjusted and raw `P(d)` share a panel so the weekday trap is visible. Zero is "the cycle's own average".

In [ ]:
LABELS = {
    "restingHr": "Resting HR (bpm)",
    "hrvMs": "HRV (ms)",
    "recovery": "Recovery (%)",
    "sleepPerformance": "Sleep performance (%)",
}
ADJUSTED_COLOR, RAW_COLOR = "#2a78d6", "#eb6834"  # categorical slots 1 and 2
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e4e3df"


def plot_profiles(result, title, weekday_of_d0=None):
    metrics = list(result["adjusted"]["profile"].index)
    fig, axes = plt.subplots(1, len(metrics), figsize=(4 * len(metrics), 3.4), sharex=True)
    fig.patch.set_facecolor("#fcfcfb")
    d = np.arange(7)
    for ax, metric in zip(axes, metrics):
        for view, color, marker in (("adjusted", ADJUSTED_COLOR, "o"), ("raw", RAW_COLOR, "s")):
            y = result[view]["profile"].loc[metric].to_numpy(dtype=float)
            ax.plot(d, y, color=color, lw=2, marker=marker, ms=5, label=view)
        ax.axhline(0, color=MUTED, lw=1)
        ax.set_title(LABELS.get(metric, metric), fontsize=10, color=INK, loc="left")
        ax.set_facecolor("#fcfcfb")
        ax.grid(axis="y", color=GRID, lw=0.8)
        for side in ("top", "right"):
            ax.spines[side].set_visible(False)
        for side in ("left", "bottom"):
            ax.spines[side].set_color(GRID)
        ax.tick_params(colors=MUTED, labelsize=8)
        ticks = [str(i) for i in d]
        if weekday_of_d0 is not None:
            ticks = [f"{i}\n{WEEKDAYS[(weekday_of_d0 + i) % 7]}" for i in d]
        ax.set_xticks(d, ticks)
        ax.set_xlabel("days since injection (d)", fontsize=8, color=MUTED)
    axes[0].legend(frameon=False, fontsize=8)
    fig.suptitle(title, x=0.01, ha="left", fontsize=11, color=INK)
    fig.tight_layout()
    plt.show()


def d0_weekday(inputs, cfg=PARAMS):
    # Weekday of the d = 0 morning for the first schedule segment (0 = Sunday).
    seg = inputs["injectionWeekdays"][0]["weekday"]
    return (seg + cfg["dayZeroRule"][inputs["injectionTime"]]) % 7

## Synthetic members (personas A, B, C)

> **Synthetic member.** These CSVs are generated by `npm run personas` from `src/config/personas.ts`. They are not real people.

- **A:** a planted within-week shape (semaglutide, Wednesday-morning injections).
- **B:** no drug effect (tirzepatide, Monday-evening injections).
- **C:** no drug effect, plus a Sunday-night habit (Sunday-morning injections, so `d = 0` is Monday). The raw view should show a fake `d = 0` effect, and the adjusted view shouldn't.

In [ ]:
def load_persona(pid):
    meta = json.loads((SYNTHETIC / f"persona_{pid.lower()}_inputs.json").read_text())
    assert meta["synthetic"] is True
    path = str(SYNTHETIC / f"persona_{pid.lower()}.csv")
    return (lambda con: run_sql(con, "01_daily_persona.sql", path=path)), meta["inputs"]


personas = {}
for pid in ("A", "B", "C"):
    loader, inputs = load_persona(pid)
    personas[pid] = (analyze(loader, inputs), inputs)

### Persona A (synthetic)

Planted pattern.

In [ ]:
result, inputs = personas["A"]
show(result)
plot_profiles(result, "Persona A (synthetic member): P(d), adjusted vs raw", d0_weekday(inputs))

### Persona B (synthetic)

No drug effect; the profile should be small noise around zero.

In [ ]:
result, inputs = personas["B"]
show(result)
plot_profiles(result, "Persona B (synthetic member): P(d), adjusted vs raw", d0_weekday(inputs))

### Persona C (synthetic)

Weekday trap: raw spikes at d = 0 (Monday), adjusted doesn't.

In [ ]:
result, inputs = personas["C"]
show(result)
plot_profiles(result, "Persona C (synthetic member): P(d), adjusted vs raw", d0_weekday(inputs))

## Cross-check: persona A against the engine

`engine_results.json` comes from `npm run export-results` (the TypeScript engine on the same persona). Point estimates, weekday means and counts, and cycle counts must match to rounding. Intervals use bootstrap standard errors, so they aren't compared.

In [ ]:
engine = json.loads((SYNTHETIC / "engine_results.json").read_text())
assert engine["synthetic"] is True and engine["persona"] == "A"
assert PARAMS == engine["config"], "PARAMS differ from src/config/analysis.ts (via engine_results.json)"

result_a = personas["A"][0]
TOL = 1e-9


def as_array(values):
    return np.array([np.nan if v is None else v for v in values], dtype=float)


rows = []
for r in engine["results"]:
    m = r["metric"]
    checks = {
        "weekday means": np.allclose(result_a["adjusted"]["weekday_means"].loc[m], as_array(r["weekdayMeans"]), atol=TOL, equal_nan=True),
        "weekday counts": list(result_a["adjusted"]["weekday_counts"].loc[m]) == r["weekdayCounts"],
        "cycles": int(result_a["adjusted"]["status"].loc[m, "cycles"]) == r["cycles"],
        "P(d) adjusted": np.allclose(result_a["adjusted"]["profile"].loc[m], as_array(r["adjusted"]), atol=TOL, equal_nan=True),
        "P(d) raw": np.allclose(result_a["raw"]["profile"].loc[m], as_array(r["raw"]), atol=TOL, equal_nan=True),
    }
    max_diff = np.nanmax(np.abs(result_a["adjusted"]["profile"].loc[m].to_numpy(dtype=float) - as_array(r["adjusted"])))
    rows.append({"metric": m, **checks, "max |diff| P(d) adjusted": max_diff})

crosscheck = pd.DataFrame(rows).set_index("metric")
display(crosscheck)
assert crosscheck.drop(columns="max |diff| P(d) adjusted").all().all(), "Notebook and engine disagree"
print("Persona A matches the engine.")

## Prakhar's WHOOP export, with a placebo first dose

Prakhar isn't on a GLP-1 drug, so this runs the method on his own data with a **made-up first dose**: semaglutide, 2026-05-04 (a Monday), morning injections, 0.25 mg. Any "pattern" here is something the method found where there's no drug. It's a real-data placebo check.

Data handling: the export stays in `notebook/data/export/` (gitignored) and nbstripout removes every output on commit. These cells print aggregates and charts only, never daily rows.

**Column mapping** (confirmed by Prakhar, 2026-10-14), from `physiological_cycles.csv`:

| DailyRow | Export column |
|---|---|
| date | "Wake onset", calendar date as written (local time) |
| cycleId | "Cycle start time" (the export has no ID) |
| restingHr | "Resting heart rate (bpm)" |
| hrvMs | "Heart rate variability (ms)" |
| recovery | "Recovery score %" |
| sleepPerformance | "Sleep performance %" |
| respiratoryRate | "Respiratory rate (rpm)" |
| skinTempC | "Skin temp (celsius)" |
| strain | "Day Strain", null while "Cycle end time" is blank |
| calibrating | wake dates 2026-01-22 to 2026-01-24 (shown as calibrating by the API) |

**Why "Wake onset" is read as local time.** The export gives no offset. "Cycle timezone" changes at the daylight-saving dates, and wake times read as local clock time stay steady across that change, while a UTC reading would make local wake times jump by an hour or more at the change. The evidence was checked on aggregates in the Phase 6 session and is deliberately not stored here. Either way, the check cell below asserts that a UTC reading would give the same wake date for every row; if that ever fails, stop and decide.

**Main sleep.** The cycles file supplies the main sleep. `sleeps.csv` rows with Nap = false are used only as a check.

In [ ]:
EXPORT_PARAMS = {
    "cycles_path": str(EXPORT / "physiological_cycles.csv"),
    "sleeps_path": str(EXPORT / "sleeps.csv"),
    "calibrating_from": "2026-01-22",
    "calibrating_to": "2026-01-24",
}

# Placebo inputs (Prakhar is not taking a GLP-1 drug).
PLACEBO_INPUTS = {
    "drug": "semaglutide",
    "firstDoseDate": "2026-05-04",
    "injectionWeekdays": [{"from": "2026-05-04", "weekday": 1}],  # Monday
    "missedInjectionDates": [],
    "movedInjections": [],
    "injectionTime": "morning",
    "doseSteps": [{"date": "2026-05-04", "mg": 0.25}],
}

In [ ]:
def load_export(con):
    run_sql(
        con,
        "01_daily_export.sql",
        cycles_path=EXPORT_PARAMS["cycles_path"],
        calibrating_from=EXPORT_PARAMS["calibrating_from"],
        calibrating_to=EXPORT_PARAMS["calibrating_to"],
    )


con = duckdb.connect()
load_export(con)
run_sql(con, "01b_sleeps_check.sql", sleeps_path=EXPORT_PARAMS["sleeps_path"])
checks = con.sql("SELECT * FROM export_checks").df()
display(checks)
display(con.sql("SELECT * FROM sleeps_check").df())
display(con.sql(
    "SELECT min(date) AS first_wake_date, max(date) AS last_wake_date, count(DISTINCT date) AS days, "
    "count(DISTINCT date) FILTER (WHERE calibrating) AS calibrating_days FROM daily"
).df())
assert int(checks.loc[0, "dates_that_differ_if_utc"]) == 0, "Wake date depends on the local/UTC reading; ask Prakhar"
assert_one_row_per_date(con)
con.close()

In [ ]:
export_result = analyze(load_export, PLACEBO_INPUTS)
show(export_result)
plot_profiles(export_result, "Prakhar's data, placebo first dose 2026-05-04: P(d), adjusted vs raw", d0_weekday(PLACEBO_INPUTS))